# J-Space Experiment — Phases 1 and 2

This thin Colab launcher runs or resumes Phase 1 layer selection and Phase 2 coarse J-space disruption using the same Python modules and command used on a CUDA machine over SSH. Scientific state is saved under one run root with `phase1/` and `phase2/` subdirectories.

## 1. Install the experiment and pinned BIPIA checkout

In [ ]:
import subprocess
from pathlib import Path

RESEARCH_REPO = 'https://github.com/ethanncyb/jspace-research.git'
RESEARCH_REVISION = 'prompt-injection-experiment'
BIPIA_REVISION = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
REPO_ROOT = Path('/content/jspace-research')
BIPIA_CHECKOUT = Path('/content/BIPIA')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', RESEARCH_REVISION, RESEARCH_REPO, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'merge', '--ff-only', f'origin/{RESEARCH_REVISION}'], check=True)
if not BIPIA_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', str(BIPIA_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(BIPIA_CHECKOUT), 'checkout', BIPIA_REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPO_ROOT)], check=True)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Authenticate

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('OpenAI judge credential loaded from Colab Secrets.')

## 3. Configure the persistent run root

In [ ]:
RUN_MODE = 'smoke'  # use 'full' only after smoke succeeds
USE_DRIVE = True
RUN_NAME = f'jspace-e2e-{RUN_MODE}'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/jspace-research/runs') / RUN_NAME
else:
    RUN_ROOT = Path('/content') / RUN_NAME

BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
WEBQA_TRAIN_PATH = None  # required for full mode
SUMMARIZATION_TRAIN_PATH = None  # required for full mode
CONFIG_PATH = REPO_ROOT / 'configs' / f'phase1_{RUN_MODE}.yaml'
print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)

## 4. Verify CUDA and run both phases

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before running the experiment.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
command = [
    'jspace-run',
    '--config', str(CONFIG_PATH),
    '--bipia-root', str(BIPIA_ROOT),
    '--run-dir', str(RUN_ROOT),
]
if WEBQA_TRAIN_PATH is not None:
    command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'End-to-end run failed with exit status {return_code}; see the traceback above.')

## 5. Inspect the frozen layer and Phase 2 results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
print(json.dumps(json.loads((PHASE1_DIR / 'selected_layer.json').read_text()), indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. It does not establish injection-specific causality.